# KATL V19-C — Dense v11 Settlement + Empirical/Ordinal Blend

This notebook implements V19-C only. It uses the same ≤3%-missing dense v11 settlement ridge backbone, then predicts the final bucket-offset class (`≤−2`, `−1`, `0`, `+1`, `≥+2`) with regularized logistic regression. The ordinal distribution is blended with V19-B empirical residual probabilities. Blend weight selection is forward/out-of-fold and cannot inspect 2026 outcomes.


In [ ]:
from pathlib import Path
import json
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src" / "calibration" / "station_stacking.py").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("Could not find the weather-research project root")
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.environ["WEATHER_RESEARCH_INCLUDE_DIRECT_NBM"] = "1"

STATION_ID = "KATL"
FAST_MODE = False
OPTUNA_TRIALS = 30
OPTUNA_STARTUP_TRIALS = 15
STACK_OPTUNA_TRIALS = 30
STACK_OPTUNA_STARTUP_TRIALS = 15
MISSINGNESS_LIMIT = 0.03
MONTHLY_SHRINKAGE = 60.0
BLEND_WEIGHTS = (0.0, 0.25, 0.5, 0.75, 1.0)
ROOT_OUTPUT = PROJECT_ROOT / "data" / "calibration" / "station_stacking_v19_patched" / STATION_ID
MODEL_OUTPUT = ROOT_OUTPUT / "dense_backbone"
METHOD_OUTPUT = ROOT_OUTPUT / "v19_c"
METHOD_OUTPUT.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT


In [ ]:
import pandas as pd

from src.calibration.station_stacking import (
    TARGET_SOURCE_SETTLEMENT_FIRST, StationStackingConfig, YearSplitFold,
    _modeling_frame, run_station_year_split_experiment,
)
from src.calibration.v19_bucket import (
    crossfit_ridge_predictions, feature_missingness_audit,
    ordinal_blend_bucket_decisions, ordinal_blend_metrics,
    paired_bootstrap_accuracy_gain,
)


## Forward contract

Feature selection, base forecasts, ridge residuals, classifier predictions, and blend selection are all forward. V19-C uses 2026 only once for final evaluation.


In [ ]:
FOLDS = (
    YearSplitFold("train_2021_valid_2022", 2021, 2021, 2022),
    YearSplitFold("train_2021_2022_valid_2023", 2021, 2022, 2023),
    YearSplitFold("train_2021_2023_valid_2024", 2021, 2023, 2024),
    YearSplitFold("train_2021_2024_valid_2025", 2021, 2024, 2025),
)
config = StationStackingConfig(
    station_id=STATION_ID, project_root=PROJECT_ROOT,
    timing_mode="same_day_11am_live_safe", providers=("gfs", "hrrr", "nbm"),
    fast_mode=FAST_MODE, optuna_trials=OPTUNA_TRIALS, stack_optuna_trials=STACK_OPTUNA_TRIALS,
    optuna_startup_trials=OPTUNA_STARTUP_TRIALS, stack_optuna_startup_trials=STACK_OPTUNA_STARTUP_TRIALS, optuna_metric="mae_f",
    optuna_verbose=True, feature_version="v11", target_mode="remaining_warmup",
    target_source=TARGET_SOURCE_SETTLEMENT_FIRST, hyperparameter_space="wide",
    base_model_methods=("xgboost", "lightgbm", "catboost"), stack_enabled=True,
    year_split_folds=FOLDS, year_split_validation_weights={2022: 1, 2023: 1, 2024: 1, 2025: 1},
    year_split_test_train_years=(2021, 2025), year_split_test_year=2026,
    max_feature_missing_fraction=MISSINGNESS_LIMIT, output_dir=MODEL_OUTPUT,
)
config


## Train or resume the shared dense backbone

V19-B and V19-C intentionally share `dense_backbone` because their temperature model is identical. Existing Optuna studies resume rather than creating a different base model.


In [ ]:
result = run_station_year_split_experiment(config)
result.scoreboard


## Audit the 3% feature gate


In [ ]:
modeling_frame, categorical, numeric = _modeling_frame(result.features, config)
missingness = feature_missingness_audit(
    modeling_frame, categorical, numeric, train_years=(2021, 2025),
    max_missing_fraction=MISSINGNESS_LIMIT,
)
missingness.to_csv(METHOD_OUTPUT / "feature_missingness_audit.csv", index=False)
missingness.groupby(["kind", "keep_v19"]).size().rename("feature_count").reset_index()


## Build cross-fitted ridge residuals


In [ ]:
residuals = crossfit_ridge_predictions(
    result.validation_predictions,
    base_model_methods=tuple(config.effective_base_model_methods),
    providers=tuple(config.providers), min_train_rows=config.effective_min_meta_train_rows,
)
residuals.to_csv(METHOD_OUTPUT / "crossfit_ridge_residuals.csv", index=False)
residuals.groupby("validation_year")["residual_f"].agg(["count", "mean", "std", "median"])


## Fit V19-C and select its blend weight

The classifier uses ridge/base/provider predictions, ensemble spreads, bucket position, and cyclic month. For each validation year it is fitted only on earlier cross-fitted years. Blend weight is selected by validation bucket log loss, with accuracy as the tie-breaker.


In [ ]:
decisions, blend_tuning, metadata = ordinal_blend_bucket_decisions(
    result.validation_predictions, result.test_predictions, residuals,
    base_model_methods=tuple(config.effective_base_model_methods),
    providers=tuple(config.providers), monthly_shrinkage=MONTHLY_SHRINKAGE,
    blend_weights=BLEND_WEIGHTS, min_train_rows=config.effective_min_meta_train_rows,
)
blend_tuning.to_csv(METHOD_OUTPUT / "ordinal_blend_tuning.csv", index=False)
(METHOD_OUTPUT / "ordinal_metadata.json").write_text(json.dumps(metadata, indent=2) + "\n", encoding="utf-8")
blend_tuning


## Final 2026 V19-C evaluation


In [ ]:
metrics = ordinal_blend_metrics(decisions)
blend_vs_empirical = paired_bootstrap_accuracy_gain(
    decisions["blended_bucket_hit"], decisions["empirical_bucket_hit"],
)
decisions.to_csv(METHOD_OUTPUT / "2026_ordinal_blend_decisions.csv", index=False)
metrics.to_csv(METHOD_OUTPUT / "2026_metrics.csv", index=False)
blend_vs_empirical.to_frame("value").to_csv(METHOD_OUTPUT / "paired_bootstrap_blend_vs_empirical.csv")
display(metrics)
display(blend_vs_empirical.to_frame("value"))


## Interpretation

A selected ordinal weight of `0.0` means validation rejected the classifier and V19-C collapses safely to V19-B. Promote the blend only if it improves pooled and station-level paired results without degrading MAE or large-miss behavior.
